# Sparse, Dense & Hybrid Retrieval with Pyversity

Pyversity is **embedding-agnostic**: its diversification algorithms only care about pairwise similarities between vectors, not how those vectors were produced. This means it works equally well with:

- **Sparse** vectors from BM25 (keyword-based term frequencies)
- **Dense** vectors from fast static models like [potion-base-32M](https://huggingface.co/minishlab/potion-base-32M)
- **Hybrid** combinations of both

This notebook demonstrates all three paradigms on the **Quora question corpus** (515k real questions from the Quora platform). We search for `"how to learn machine learning"` — a query that returns many near-identical question phrasings without diversification, and distinct subtopics with it.

In [ ]:
%pip install pyversity bm25s sentence-transformers datasets

In [ ]:
import numpy as np
import scipy.sparse as sp
import bm25s
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pyversity import diversify, Strategy

## The Corpus

We use the **BEIR Quora corpus** — 515,000 real questions from the Quora platform, covering an enormous range of topics.

**Query:** `"how to learn machine learning"`

This is a deliberately redundancy-prone query: without diversification, the top results are near-identical phrasings of the same question. Diversification surfaces distinct subtopics (learning for economists, learning statistics, learning as a programmer, etc.).

In [ ]:
# Load the Quora corpus from BEIR (~15 MB, cached automatically after first download)
# 515,000 real questions from the Quora platform.
print("Downloading Quora corpus...")
ds = load_dataset("BeIR/quora", "corpus", split="corpus")

corpus = [
    (doc["title"] + " " + doc["text"]).strip()
    for doc in ds
    if len((doc["title"] + " " + doc["text"]).strip()) > 20
]

query = "how to learn machine learning"
K = 20   # candidates to retrieve
k = 5    # items to select after diversification

print(f"Corpus size : {len(corpus):,} questions")
print(f"Query       : '{query}'")
print(f"Retrieve top-{K}, then diversify to {k}")

---
## Part 1: Sparse Retrieval with BM25

BM25 is a classic keyword-matching algorithm that scores documents based on term frequency and inverse document frequency, producing **sparse** term vectors — most entries are zero because most words do not appear in a given document.

Pyversity only needs an array of shape `(n_candidates, n_features)`. For BM25, we reconstruct the internal score matrix and slice to the top-K candidates. A single `.toarray()` call converts the scipy sparse matrix to a dense NumPy array that pyversity can consume.

> The primary point of this section is the API bridge: pyversity works with any NumPy array, sparse or dense. The diversity effect from BM25 term vectors is modest — they capture vocabulary overlap rather than semantic similarity, so they are a coarser signal for redundancy than dense embeddings.

In [4]:
# Index the full 515k-question corpus with BM25
corpus_tokens = bm25s.tokenize(corpus)
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# Retrieve top-K candidates
query_tokens = bm25s.tokenize([query])
results, scores = retriever.retrieve(query_tokens, k=K)

bm25_candidate_indices = results[0].tolist()   # top-K indices into corpus
bm25_candidate_scores  = scores[0]             # BM25 relevance scores

print("Top-K BM25 candidates:")
for rank, (idx, score) in enumerate(zip(bm25_candidate_indices, bm25_candidate_scores), 1):
    print(f"  {rank:2d}. (score={score:.2f}) {corpus[idx]}")

Split strings:   0%|          | 0/514930 [00:00<?, ?it/s]

Split strings:   5%|▌         | 27862/514930 [00:00<00:01, 278601.00it/s]

Split strings:  11%|█▏        | 58611/514930 [00:00<00:01, 295578.47it/s]

Split strings:  17%|█▋        | 88169/514930 [00:00<00:01, 290693.23it/s]

Split strings:  23%|██▎       | 117248/514930 [00:00<00:01, 226057.99it/s]

...

BM25S Compute Scores:  93%|█████████▎| 480998/514930 [00:02<00:00, 186702.02it/s]

BM25S Compute Scores:  97%|█████████▋| 499681/514930 [00:02<00:00, 185583.43it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Top-K BM25 candidates:
   1. (score=8.28) How can I learn machine learning?
   2. (score=8.28) How do I learn machine learning?
   3. (score=8.16) Should I learn machine learning?
   4. (score=7.76) How do I learn machine learning as a programmer?
   5. (score=7.76) How can I learn machine learning well?
   6. (score=7.76) How can I learn machine learning better?
   7. (score=7.76) How do I learn mathematics for machine learning?
   8. (score=7.61) Should economists learn machine learning?
   9. (score=7.30) How do I learn Machine Learning in 10 days?
  10. (score=7.30) How do I learn machine learning and from where?
  11. (score=7.30) How do I learn statistics and probability for machine learning?
  12. (score=6.92) What is machine to machine learning?
  13. (score=6.71) Can a person with no Coding knowledge learn Machine learning?
  14. (score=6.71) What do I need to know to learn machine learning?
  15. (score=6.71) Why should I learn the math behind machine learning?
  16. (score=6

In [5]:
# bm25s stores its term scores as raw CSR/CSC arrays.
# We reassemble them into a scipy sparse matrix (docs × vocab),
# then slice to our K candidates and call .toarray() —
# the only step needed to bridge sparse BM25 vectors into pyversity.
s = retriever.scores
bm25_score_matrix = sp.csc_matrix(
    (np.array(s["data"]), np.array(s["indices"]), np.array(s["indptr"])),
    shape=(int(s["num_docs"]), len(s["indptr"]) - 1),
)  # shape: (n_docs, vocab_size)

bm25_candidate_embeddings = bm25_score_matrix[bm25_candidate_indices].toarray()  # (K, vocab_size)

print(f"Sparse BM25 matrix shape : {bm25_score_matrix.shape}")
print(f"Candidate embeddings     : {bm25_candidate_embeddings.shape}  (K × vocab_size)")
print(f"Sparsity                 : {(bm25_candidate_embeddings == 0).mean():.1%} zeros")

Sparse BM25 matrix shape : (514930, 85392)
Candidate embeddings     : (20, 85392)  (K × vocab_size)
Sparsity                 : 100.0% zeros


In [6]:
# Naive top-5: diversity=0.0 → pure relevance ranking, no diversification
naive_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== BM25 — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_bm25.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== BM25 — Naive top-5 (diversity=0.0) ===
  1. How can I learn machine learning?
  2. How do I learn machine learning?
  3. Should I learn machine learning?
  4. How do I learn machine learning as a programmer?
  5. How can I learn machine learning well?


In [7]:
# Diversified top-5 with DPP
diverse_bm25 = diversify(
    embeddings=bm25_candidate_embeddings,
    scores=bm25_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== BM25 — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_bm25.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== BM25 — Diversified top-5 (diversity=0.7) ===
  1. How can I learn machine learning?
  2. Should economists learn machine learning?
  3. How do I learn machine learning as a programmer?
  4. How do I learn mathematics for machine learning?
  5. How do I learn statistics and probability for machine learning?


---
## Part 2: Dense Re-ranking with Static Embeddings

Dense embeddings capture **semantic meaning** — "optimization" and "minimization" end up close even without shared tokens. Here we re-score the same BM25 top-K candidates using a dense encoder, then diversify based on dense similarity.

We use [**potion-base-32M**](https://huggingface.co/minishlab/potion-base-32M), a fast static embedding model from the [model2vec](https://github.com/MinishLab/model2vec) family. It is orders of magnitude faster than transformer-based encoders (no GPU needed) while retaining strong retrieval quality.

> This 2-stage pattern — BM25 for fast first-stage retrieval, dense model for re-ranking — is common in production systems.

In [8]:
# Encode only the 20 BM25 candidates and the query with potion-base-32M.
# This 2-stage pattern (BM25 retrieval → dense re-ranking) is common in production.
# device="cpu" — static embedding models do not benefit from GPU/MPS
model = SentenceTransformer("minishlab/potion-base-32M", device="cpu")

bm25_candidate_texts       = [corpus[i] for i in bm25_candidate_indices]
query_embedding            = model.encode([query], normalize_embeddings=True)
dense_candidate_embeddings = model.encode(bm25_candidate_texts, normalize_embeddings=True)

# Cosine similarity (L2-normalised vectors → dot product = cosine sim)
dense_candidate_scores = (query_embedding @ dense_candidate_embeddings.T)[0]

print("BM25 candidates re-scored by dense model:")
for rank, i in enumerate(np.argsort(dense_candidate_scores)[::-1], 1):
    print(f"  {rank:2d}. (score={dense_candidate_scores[i]:.3f}) {corpus[bm25_candidate_indices[i]]}")

BM25 candidates re-scored by dense model:
   1. (score=0.989) How can I learn machine learning?
   2. (score=0.971) How do I learn machine learning?
   3. (score=0.956) How can I learn machine learning well?
   4. (score=0.930) Should I learn machine learning?
   5. (score=0.929) How do I learn machine learning and from where?
   6. (score=0.925) How can I learn machine learning better?
   7. (score=0.895) What do I need to know to learn machine learning?
   8. (score=0.886) How do I start learning machine learning?
   9. (score=0.879) How do I learn mathematics for machine learning?
  10. (score=0.861) How do I learn Machine Learning in 10 days?
  11. (score=0.838) How do I learn machine learning as a programmer?
  12. (score=0.801) Can a person with no Coding knowledge learn Machine learning?
  13. (score=0.799) What is machine to machine learning?
  14. (score=0.781) What are the good books and resources to learn machine learning?
  15. (score=0.772) what steps should I follow to le

In [9]:
# Naive top-5: no diversification — all results are near-identical question phrasings
naive_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Dense — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Dense — Naive top-5 (diversity=0.0) ===
  1. How can I learn machine learning?
  2. How do I learn machine learning?
  3. How can I learn machine learning well?
  4. Should I learn machine learning?
  5. How do I learn machine learning and from where?


In [10]:
# Diversified top-5: distinct subtopics surface (economics, statistics, programming, etc.)
diverse_dense = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Dense — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Dense — Diversified top-5 (diversity=0.7) ===
  1. How can I learn machine learning?
  2. How do I start learning machine learning?
  3. How do I learn Machine Learning in 10 days?
  4. What do I need to know to learn machine learning?
  5. What is machine to machine learning?


In [11]:
# Side-by-side: the contrast between naive and diversified is immediately visible
diverse_dense_full = diversify(
    embeddings=dense_candidate_embeddings,
    scores=dense_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=1.0,
)

print("Naive (d=0.0)   — all 5 ask essentially the same thing:")
for rank, i in enumerate(naive_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

print()
print("Diversified (d=0.7) — varied perspectives on learning ML:")
for rank, i in enumerate(diverse_dense.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

print()
print("Maximum diversity (d=1.0) — maximally distinct subtopics:")
for rank, i in enumerate(diverse_dense_full.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

Naive (d=0.0)   — all 5 ask essentially the same thing:
  1. How can I learn machine learning?
  2. How do I learn machine learning?
  3. How can I learn machine learning well?
  4. Should I learn machine learning?
  5. How do I learn machine learning and from where?

Diversified (d=0.7) — varied perspectives on learning ML:
  1. How can I learn machine learning?
  2. How do I start learning machine learning?
  3. How do I learn Machine Learning in 10 days?
  4. What do I need to know to learn machine learning?
  5. What is machine to machine learning?

Maximum diversity (d=1.0) — maximally distinct subtopics:
  1. How can I learn machine learning?
  2. How is machine learning in asu?
  3. Why is it important to learn gradient descent in machine learning?
  4. How do I learn statistics and probability for machine learning?
  5. what steps should I follow to learn machine learning?


---
## Part 3: Hybrid Re-ranking

Hybrid retrieval fuses BM25 and dense scores. We normalise each distribution to `[0, 1]` and linearly interpolate, then diversify using dense embeddings.

Two separate concerns:
- **Hybrid score** (BM25 + dense) — determines *which* candidates rank highest. Fusing both signals can improve ranking: BM25 rewards exact keyword matches, dense rewards semantic neighbours.
- **Dense embeddings** for diversity — determines *how redundant* two candidates are. BM25 term vectors only reflect vocabulary overlap with the query, not passage-to-passage semantic similarity, so dense embeddings are always the right tool for the diversity step.

In [12]:
# Fuse BM25 and dense scores for the same K candidates
def min_max_normalize(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

alpha = 0.5
hybrid_candidate_scores = (
    alpha * min_max_normalize(dense_candidate_scores)
    + (1 - alpha) * min_max_normalize(bm25_candidate_scores)
)

print("BM25 candidates re-scored by hybrid (BM25 + dense):")
for rank, i in enumerate(np.argsort(hybrid_candidate_scores)[::-1], 1):
    print(f"  {rank:2d}. (score={hybrid_candidate_scores[i]:.3f}) {corpus[bm25_candidate_indices[i]]}")

BM25 candidates re-scored by hybrid (BM25 + dense):
   1. (score=1.000) How can I learn machine learning?
   2. (score=0.974) How do I learn machine learning?
   3. (score=0.880) Should I learn machine learning?
   4. (score=0.795) How can I learn machine learning well?
   5. (score=0.753) How can I learn machine learning better?
   6. (score=0.689) How do I learn mathematics for machine learning?
   7. (score=0.633) How do I learn machine learning as a programmer?
   8. (score=0.619) How do I learn machine learning and from where?
   9. (score=0.526) How do I learn Machine Learning in 10 days?
  10. (score=0.476) Should economists learn machine learning?
  11. (score=0.392) What do I need to know to learn machine learning?
  12. (score=0.378) How do I start learning machine learning?
  13. (score=0.371) How do I learn statistics and probability for machine learning?
  14. (score=0.324) What is machine to machine learning?
  15. (score=0.263) Can a person with no Coding knowledge learn

In [13]:
naive_hybrid = diversify(
    embeddings=dense_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.0,
)

print("=== Hybrid — Naive top-5 (diversity=0.0) ===")
for rank, i in enumerate(naive_hybrid.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Hybrid — Naive top-5 (diversity=0.0) ===
  1. How can I learn machine learning?
  2. How do I learn machine learning?
  3. Should I learn machine learning?
  4. How can I learn machine learning well?
  5. How can I learn machine learning better?


In [14]:
diverse_hybrid = diversify(
    embeddings=dense_candidate_embeddings,
    scores=hybrid_candidate_scores,
    k=k,
    strategy=Strategy.DPP,
    diversity=0.7,
)

print("=== Hybrid — Diversified top-5 (diversity=0.7) ===")
for rank, i in enumerate(diverse_hybrid.indices, 1):
    print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

=== Hybrid — Diversified top-5 (diversity=0.7) ===
  1. How can I learn machine learning?
  2. Should economists learn machine learning?
  3. How do I learn machine learning as a programmer?
  4. How do I learn statistics and probability for machine learning?
  5. How do I learn mathematics for machine learning?


---
## Part 4: Comparing the Three Approaches

Diversified top-5 across all three retrieval methods, for the same query on the same corpus.

In [15]:
print("Diversified top-5 across all three retrieval methods")
print("=" * 90)

for label, result in [
    ("BM25 (sparse)",   diverse_bm25),
    ("Dense (potion)",  diverse_dense),
    ("Hybrid",          diverse_hybrid),
]:
    print(f"\n{label}:")
    for rank, i in enumerate(result.indices, 1):
        print(f"  {rank}. {corpus[bm25_candidate_indices[i]]}")

Diversified top-5 across all three retrieval methods

BM25 (sparse):
  1. How can I learn machine learning?
  2. Should economists learn machine learning?
  3. How do I learn machine learning as a programmer?
  4. How do I learn mathematics for machine learning?
  5. How do I learn statistics and probability for machine learning?

Dense (potion):
  1. How can I learn machine learning?
  2. How do I start learning machine learning?
  3. How do I learn Machine Learning in 10 days?
  4. What do I need to know to learn machine learning?
  5. What is machine to machine learning?

Hybrid:
  1. How can I learn machine learning?
  2. Should economists learn machine learning?
  3. How do I learn machine learning as a programmer?
  4. How do I learn statistics and probability for machine learning?
  5. How do I learn mathematics for machine learning?


---
## Key Takeaways

1. **Pyversity is embedding-agnostic.** DPP only operates on pairwise similarities between vectors — it does not matter whether those vectors came from BM25, a static model, a transformer, or anything else that produces a NumPy array.

2. **`.toarray()` is all you need for sparse vectors.** BM25 score matrices are scipy sparse arrays. One call converts them to a dense NumPy array that pyversity accepts.

3. **Dense embeddings unlock richer diversity.** Semantic similarity between candidates (not just keyword overlap) gives diversification a stronger signal, surfacing genuinely distinct subtopics rather than just different phrasings.

4. **The `diversity` parameter is a dial.** `0.0` = pure relevance ranking. `1.0` = maximum topic coverage. Values of `0.5–0.8` give the best relevance/diversity trade-off for most production use cases.